In [2]:
import numpy as np
import pandas as pd
import torch

from chronos import ChronosPipeline

from gluonts.dataset.pandas import (
    PandasDataset
)

from tsfm_public import (
    TinyTimeMixerForPrediction
)

In [3]:
%run 00.ipynb

c:\Users\SafinSharma\miniconda3\envs\foundation\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Export Folder : standardized_exports
Results Folder: results
24
Datasets Found: 22
agri_medium
agri_small
energy_medium
energy_small
favorita-train_medium
favorita-train_medium_cov
favorita-train_small
favorita-train_small_cov
favorita-transaction_medium
favorita-transaction_small
m4-daily_medium
m4-daily_small
m4-hourly_medium
m4-hourly_small
m4-quarterly_medium
m4-quarterly_small
m4-weekly_medium
m4-weekly_small
m4-yearly_medium
m4-yearly_small
traffic_medium
traffic_small
5
agri_medium               rows=15343      series=246      horizon=5
5
agri_small                rows=14428      series=246      horizon=5
24
energy_medium             rows=177600     series=370      horizon=24
24
energy_small              rows=44400      series=370      horizon=24
28
favorita-train_medium     rows=1985155    series=4504     horizon=28
favorita-train_medium_cov -> ERROR: No horizon configured for favorita-train_medium_cov
28
favorita-train_small      rows=609961     series=4504     horizon=28
favo

In [4]:
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

Device: cpu


In [5]:
RESULT_FILE = "foundation_results.csv"

initialize_results_file(
    RESULT_FILE
)

completed_runs = get_completed_runs(
    RESULT_FILE
)

print(
    f"Completed Runs: {len(completed_runs)}"
)

Completed Runs: 16


In [6]:
chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=DEVICE
)

In [7]:
def chronos_forecast(
    train_series,
    horizon
):

    context = torch.tensor(
        train_series.values,
        dtype=torch.float32
    )

    forecast = chronos_model.predict(

        context,

        prediction_length=horizon,

        num_samples=20
    )

    return (
        forecast
        .median(dim=1)
        .values
        .cpu()
        .numpy()
        .flatten()
    )

In [8]:
ttm_model = (
    TinyTimeMixerForPrediction
    .from_pretrained(
        "ibm-research/ttm-r3"
    )
)

ttm_model.eval()

Some weights of TinyTimeMixerForPrediction were not initialized from the model checkpoint at ibm-research/ttm-r3 and are newly initialized: ['backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.gating_block.attn_layer.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.gating_block.attn_layer.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc1.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc1.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc2.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc2.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.norm.norm.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.norm.norm.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.patch_mixer.gating_block.attn_layer.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.patch_mixer.gating_block.attn_layer.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.

TinyTimeMixerForPrediction(
  (backbone): TinyTimeMixerModel(
    (encoder): TinyTimeMixerEncoder(
      (patcher): Linear(in_features=9, out_features=128, bias=True)
      (mlp_mixer_encoder): TinyTimeMixerBlock(
        (mixers): ModuleList(
          (0-9): 10 x TinyTimeMixerLayer(
            (patch_mixer): PatchMixerBlock(
              (norm): TinyTimeMixerNormLayer(
                (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
              )
              (mlp): TinyTimeMixerMLP(
                (fc1): Linear(in_features=56, out_features=168, bias=True)
                (dropout1): Dropout(p=0.2, inplace=False)
                (fc2): Linear(in_features=168, out_features=56, bias=True)
                (dropout2): Dropout(p=0.2, inplace=False)
              )
              (gating_block): TinyTimeMixerGatedAttention(
                (attn_layer): Linear(in_features=56, out_features=56, bias=True)
                (attn_softmax): Softmax(dim=-1)
          

In [9]:
def ttm_forecast(
    train_series,
    horizon
):

    x = torch.tensor(
        train_series.values,
        dtype=torch.float32
    )

    x = x.unsqueeze(0)

    with torch.no_grad():

        output = ttm_model(
            past_values=x
        )

    pred = (
        output
        .prediction_outputs
        .flatten()
        .cpu()
        .numpy()
    )

    return pred[:horizon]

In [10]:
def evaluate_foundation_model(
    train_df,
    test_df,
    horizon,
    forecast_function
):

    actuals = []
    forecasts = []

    for uid, train_group in train_df.groupby(
        "unique_id"
    ):

        test_group = test_df[
            test_df["unique_id"] == uid
        ]

        if len(test_group) == 0:
            continue

        train_series = (
            train_group["y"]
            .astype(float)
        )

        try:

            pred = forecast_function(
                train_series,
                horizon
            )

        except:

            pred = np.repeat(
                train_series.iloc[-1],
                horizon
            )

        actuals.extend(
            test_group["y"].values
        )

        forecasts.extend(
            pred[:len(test_group)]
        )

    metrics = calculate_metrics(
        actuals,
        forecasts
    )

    return metrics

In [ ]:
MODELS = {

    "Chronos":
    chronos_forecast,

    "TTM":
    ttm_forecast
}

In [12]:
for dataset_name in AVAILABLE_DATASETS:

    print("\n" + "="*80)
    print(dataset_name)

    try:

        (
            df,
            train_df,
            test_df,
            horizon
        ) = prepare_dataset(
            dataset_name
        )

        for model_name, forecast_fn in MODELS.items():

            if (
                dataset_name,
                model_name
            ) in completed_runs:

                print(
                    f"SKIP : {model_name}"
                )

                continue

            print(
                f"RUN : {model_name}"
            )

            start = start_timer()

            metrics = (
                evaluate_foundation_model(
                    train_df,
                    test_df,
                    horizon,
                    forecast_fn
                )
            )

            runtime = stop_timer(
                start
            )

            result = {

                "model":
                model_name,

                "dataset":
                dataset_name,

                "n_series":
                df["unique_id"].nunique(),

                "n_rows":
                len(df),

                "horizon":
                horizon,

                "runtime_seconds":
                runtime,

                **metrics
            }

            append_result(
                RESULT_FILE,
                result
            )

            print(
                f"MAE={metrics['MAE']:.4f}"
            )

    except Exception as e:

        print(
            f"ERROR : {e}"
        )


agri_medium
5
SKIP : TTM

agri_small
5
SKIP : TTM

energy_medium
24
SKIP : TTM

energy_small
24
SKIP : TTM

favorita-train_medium
28
SKIP : TTM

favorita-train_medium_cov
ERROR : No horizon configured for favorita-train_medium_cov

favorita-train_small
28
SKIP : TTM

favorita-train_small_cov
ERROR : No horizon configured for favorita-train_small_cov

favorita-transaction_medium
28
SKIP : TTM

favorita-transaction_small
28
SKIP : TTM

m4-daily_medium
20
RUN : TTM
MAE=196.0379

m4-daily_small
20
RUN : TTM
MAE=196.0379

m4-hourly_medium
20
RUN : TTM
MAE=1787.0354

m4-hourly_small
20
RUN : TTM
MAE=1787.0354

m4-quarterly_medium
20
RUN : TTM
MAE=976.9549

m4-quarterly_small
20
RUN : TTM
MAE=976.9549

m4-weekly_medium
20
RUN : TTM
MAE=322.1104

m4-weekly_small
20
RUN : TTM
MAE=322.1104

m4-yearly_medium
20
RUN : TTM
MAE=1681.8366

m4-yearly_small
20
RUN : TTM
MAE=1681.8366

traffic_medium
48
RUN : TTM
MAE=4.3812

traffic_small
48
RUN : TTM
MAE=4.3812


In [13]:
results_df = pd.read_csv(
    RESULTS_FOLDER /
    RESULT_FILE
)

results_df

,model,dataset,n_series,n_rows,horizon,runtime_seconds,MAE,RMSE,SMAPE
0,Chronos,agri_medium,246,15343,5,157.893263,611568.533673,3.953333e+06,11.334846
1,TTM,agri_medium,246,15343,5,0.600201,590406.644935,3.723966e+06,10.800519
2,Chronos,agri_small,246,14428,5,148.308840,610341.840072,4.391451e+06,11.494348
3,TTM,agri_small,246,14428,5,0.584656,590406.644935,3.723966e+06,10.800519
4,Chronos,energy_medium,370,177600,24,2042.071710,150.934506,5.864700e+02,30.428595
5,TTM,energy_medium,370,177600,24,2.534525,216.590254,9.350066e+02,40.821494
6,Chronos,energy_small,370,44400,24,1428.527211,150.234378,5.551120e+02,31.336106
7,TTM,energy_small,370,44400,24,1.953196,216.590254,9.350066e+02,40.821494
8,Chronos,favorita-train_medium,4504,1985155,28,22274.703960,3.155883,1.179996e+01,50.717845
9,TTM,favorita-train_medium,4504,1985155,28,50.608117,4.029505,1.288459e+01,59.088928


In [14]:
results_df.groupby(
    "model"
)[
    [
        "MAE",
        "RMSE",
        "SMAPE",
        "runtime_seconds"
    ]
].mean().round(4)

,MAE,RMSE,SMAPE,runtime_seconds
model,,,,
Chronos,152803.5878,1.043283e+06,24.8993,4553.9464
TTM,59569.3050,3.742693e+05,25.5514,154.3657
